In [6]:
!pip install hmmlearn
!pip install fredapi

In [2]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.linear_model import LinearRegression
import cvxpy as cp
import scipy.stats as stats
from hmmlearn.hmm import GaussianHMM

In [9]:
@dataclass
class Universe:
  Bonds:List[str]
  ManagedFutures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

@dataclass
class OptimParams:
  es_prct: float
  turnover_penalty: float
  risk_penalty: float
  tail_penalty: float

@dataclass
class FilterParams:
  corr_threshold:float=0.8
  rfr:float=0.003,
  max_r2:float=0.15,
  min_sharpe:float=0.30,
  min_iur:float=0.70,
  T:int=252
  max_equity_corr: float = 0.35,
  min_cpi_corr: float = 0.20,
  max_beta_std: float = 0.25

@dataclass
class HMMParams:
    n_components: int = 2
    n_iter: int = 50
    tol: float = 1e-4
    init_params: str = "stmc"
    covariance_type: str = "diag"




In [10]:
@dataclass
class PortfolioDist:
  core:List[str]
  satellite:List[str]

In [13]:
 dist = PortfolioDist(
  core = ["Bonds", "Futures", "Commodities"],
  satellite = ["High_Beta", "High_Yield", "Sat_Defensive"]
 )

In [11]:
test_universe = Universe(
    # Core: Duration & Real Yields
    Bonds=[
        "TLT",  # iShares 20+ Year Treasury Bond ETF (Long Duration)
        "IEF",  # iShares 7-10 Year Treasury Bond ETF (Intermediate Duration)
        "TIP"   # iShares TIPS Bond ETF (Inflation-Protected)
    ],

    # Core: Convexity & Systemic Trend Protection
    ManagedFutures=[
        "DBMF", # iMGP DBi Managed Futures Strategy ETF (Multi-Asset Trend)
        "KMLM", # KFA Mount Lucas Managed Futures Index ETF (Commodity/FX/Rate Trend)
        "CTA"   # Simplify Managed Futures Strategy ETF (Absolute Return / Rates Trend)
    ],

    # Core: Real Assets & Inflation Hedges
    Commodities=[
        "PDBC", # Invesco Optimum Yield Diversified Commodity Strategy No-K1 ETF
        "GLD",  # SPDR Gold Shares (Safe Haven / Real Rates)
        "USO"   # United States Oil Fund (Energy Risk Factor)
    ],

    # Satellite: High-Beta / Growth / Cyclicals
    High_Beta=[
        "XLK",  # Technology Select Sector SPDR Fund (Mega-Cap Tech)
        "XBI",  # SPDR S&P Biotech ETF (Equal-Weighted High-Beta Biotech)
        "XLF"   # Financial Select Sector SPDR Fund (Banking / Rates Sensitivity)
    ],

    # Satellite: Credit Spreads & High Yield
    High_Yield=[
        "HYG",  # iShares iBoxx $ High Yield Corporate Bond ETF (US Junk)
        "JNK",  # SPDR Bloomberg High Yield Bond ETF (High-Yield Credit)
        "EMB"   # iShares J.P. Morgan USD Emerging Markets Bond ETF (EM Credit)
    ],

    # Satellite: Defensive Cash-Flow Anchor
    Sat_Defensive=[
        "XLP",  # Consumer Staples Select Sector SPDR Fund (Defensive Value)
        "XLU",  # Utilities Select Sector SPDR Fund (Low Beta / Yield)
        "XLV"   # Health Care Select Sector SPDR Fund (Defensive Growth)
    ]
)

In [16]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers = list(universe.values())
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      tickers.append(benchmark)
      df = yf.download(tickers, start, end, interval, group_by="tickers")

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]["Close"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark

  def get_cpi(self, start, end, api_key):
    from fredapi import Fred
    fred = Fred(api_key)

    return None


  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [5]:
class Filter:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def corr_filter(self, returns):
    corr_matrix = returns.corr()
    std = returns.std()

    sharpe = (np.mean(returns)-self.rf) / std

    drop = []
    for ticker in returns.columns:
      if ticker in drop:
        continue
      ticker_idx = returns.columns.get_loc(ticker)

      for i in returns.columns:
        if ticker == i or i in drop:
          continue

        i_idx = returns.columns.get_loc(i)
        if corr_matrix.iloc[ticker_idx, i_idx] > self.corr_threshold:
          if self.debug:
            print(corr_matrix.iloc[ticker_idx, i_idx])
          if sharpe[ticker] > sharpe[i]:
            drop.append(i)
          else:
            drop.append(ticker)

    tmp = returns.drop(columns=drop)

    return returns.columns

  def abs_return_unq_filter(
    self,
    returns:pd.DataFrame,
    benchmark:pd.Series,
    rfr:float=0.003,
    max_r2:float=0.15,
    min_sharpe:float=0.30,
    min_iur:float=0.70,
    T:int=252
  )->list:
    model, X, y = self._run_regression(
        [returns, benchmark]
    )
    beta = model.coef_[0]
    r2 = model.score(X, y)

    pred = model.predict(X)
    residuals = y - pred

    iur = np.var(residuals) / np.var(y) if np.var(y) > 0 else 0

    ann_res_mean = np.mean(residuals) * 252
    ann_res_std = np.std(residuals) * np.sqrt(252)
    residual_sharpe = ann_res_mean / ann_res_std if ann_res_std > 0 else -np.inf

    passed_r2 = r2 <= max_r2
    passed_iur = iur >= min_iur
    passed_sharpe = residual_sharpe >= min_sharpe

    is_qualified = passed_r2 and passed_iur and passed_sharpe

    return is_qualified

  def _run_regression(self, asset_list):
    df = pd.concat(
      asset_list,
      axis=1
    ).dropna()
    df.columns = ['asset', 'benchmark']

    X = df[['benchmark']].values
    y = df['asset'].values

    model = LinearRegression().fit(X, y)

    return model, X, y

  def commodity_factor_filter(
      asset_returns: pd.Series,
      market_returns: pd.Series,
      cpi_surprises: pd.Series,
      max_equity_corr: float = 0.35,
      min_cpi_corr: float = 0.20,
      max_beta_std: float = 0.25
  ) -> list:
    df = pd.concat([asset_returns, market_returns, cpi_surprises], axis=1).dropna()
    df.columns = ['Asset', 'Market', 'CPI_Surprise']

    equity_corr = df['Asset'].corr(df['Market'])
    pass_equity = abs(equity_corr) <= max_equity_corr

    cpi_corr = df['Asset'].corr(df['CPI_Surprise'])
    pass_cpi = cpi_corr >= min_cpi_corr

    rolling_cov = df['Asset'].rolling(60).cov(df['Market'])
    rolling_var = df['Market'].rolling(60).var()
    rolling_beta = (rolling_cov / rolling_var).dropna()
    beta_std = rolling_beta.std()
    pass_beta = beta_std <= max_beta_std

    is_qualified = pass_equity and pass_cpi and pass_beta

    return is_qualified


  def high_beta_filter(
    self,
    returns:pd.DataFrame,
    benchmark:pd.DataFrame,
    min_sector_beta:float=1.2
  ):
    model, X, y = self._run_regression(
        [returns, benchmark]
    )
    beta = model.coef_[0]
    passed_beta = beta > min_sector_beta

    return passed_beta

  def downside_capture_filter(self, returns, benchmark):
    down_mask = returns < 0

    if not down_mask.any():
        return 1.0

    asset_down_compound = np.prod(1 + returns[down_mask]) - 1
    bench_down_compound = np.prod(1 + benchmark[down_mask]) - 1

    return asset_down_compound / bench_down_compound


In [21]:
class HMMClassifier:
  def __init__(self, hmm_params, debug=False, **kwargs):
    self.debug = debug
    hmm_params = asdict(hmm_params) if hasattr(hmm_params, '__dataclass_fields__') else hmm_params
    self.hmm = GaussianHMM(**hmm_params)
    self.trained = False

  def fit(self, X):
    self.trained = True
    return self.hmm.fit(X)

  def get_regime_probs(self, X):
    if not self.trained:
      raise ValueError("HMM Classifier not trained")

    return self.hmm.predict_proba(X)

In [38]:
class RegimeClassifier(HMMClassifier):
  def __init__(self, hmm_params, debug=False, **kwargs):
    super().__init__(hmm_params=hmm_params, debug=debug, **kwargs)
    self.s_mean = None
    self.s_std = None
    self.eff_dim_mean = None
    self.eff_dim_std = None
    self.vol_mean = None
    self.vol_std = None
    self._train_returns = None
    self._train_market = None

  def _apply_ledoit_wolf(self, r_norm, gram_sample, T, n_assets):
    mean_mkt_corr = (np.sum(gram_sample) - n_assets) / (n_assets * (n_assets - 1))
    target = np.full((n_assets, n_assets), mean_mkt_corr)
    np.fill_diagonal(target, 1.0)

    noise_mtx = 0.0
    for t in range(T):
      r_t = r_norm[t, :].reshape(-1, 1)
      sample_t_mtx = r_t @ r_t.T
      noise_mtx += np.sum((sample_t_mtx - gram_sample) ** 2)

    noise = noise_mtx / (T ** 2)
    dist  = np.sum((gram_sample - target) ** 2)

    if dist == 0:
      return gram_sample

    shrinkage_intensity = np.clip(noise / dist, 0.0, 1.0)
    return shrinkage_intensity * target + (1 - shrinkage_intensity) * gram_sample

  def calculate_spec_ent(self, eig_vals):
    regime_probs = eig_vals / np.sum(eig_vals)
    regime_probs = np.clip(regime_probs, 1e-12, 1.0)
    return -np.sum(regime_probs * np.log(regime_probs))

  def _get_eff_dim(self, r_norm, T):
    G = (1 / T) * (r_norm.T @ r_norm)
    n_assets = G.shape[0]
    G_cond   = self._apply_ledoit_wolf(r_norm, G, T, n_assets)
    eig_vals = np.clip(np.linalg.eigvalsh(G_cond), a_min=1e-12, a_max=None)
    spec_ent = self.calculate_spec_ent(eig_vals)
    return np.exp(spec_ent)

  def garman_klass_vol(self, market, lambda_param=0.94):
    ln_CO = np.log(market["Close"] / market["Open"])
    ln_HL = np.log(market["High"] / market["Low"])
    gk_var = 0.5 * (ln_HL ** 2) - (2 * np.log(2) - 1) * (ln_CO ** 2)

    T = len(gk_var)
    smoothed_variance    = np.zeros(T)
    smoothed_variance[0] = gk_var.iloc[0]
    for t in range(1, T):
      smoothed_variance[t] = (lambda_param * smoothed_variance[t - 1]) + ((1 - lambda_param) * gk_var.iloc[t])
    return np.sqrt(smoothed_variance * 252)

  def _get_rolling_eff_dim(self, r_norm, lookback: int = 60):
      T, n_assets = r_norm.shape
      eff_dim_series = np.full(T, np.nan)
      min_obs = max(10, n_assets + 2)

      for t in range(T):
        start = max(0, t - lookback + 1)
        window = r_norm[start:t + 1, :]
        if window.shape[0] < min_obs:
          continue
        eff_dim_series[t] = self._get_eff_dim(window, window.shape[0])

      valid_mask = ~np.isnan(eff_dim_series)
      if not valid_mask.any():
        raise ValueError("Not enough observations to compute effective dimension.")

      first_valid = np.argmax(valid_mask)
      eff_dim_series[:first_valid] = eff_dim_series[first_valid]
      return eff_dim_series

  def fit_transform_features(self, sector_returns, market_ohlc, eff_dim_lookback=60):
    s_r_log = np.log(1.0 + sector_returns).values

    self.s_mean = np.mean(s_r_log, axis=0)
    self.s_std = np.std(s_r_log, axis=0, ddof=0)
    self.s_std[self.s_std == 0] = 1.0

    s_r_norm = (s_r_log - self.s_mean) / self.s_std
    eff_dim_series = self._get_rolling_eff_dim(s_r_norm, lookback=eff_dim_lookback)
    vol = self.garman_klass_vol(market_ohlc)

    self.eff_dim_mean, self.eff_dim_std = np.mean(eff_dim_series), np.std(eff_dim_series, ddof=0)
    self.vol_mean, self.vol_std = np.mean(vol), np.std(vol, ddof=0)

    eff_dim_norm = (eff_dim_series - self.eff_dim_mean) / (self.eff_dim_std if self.eff_dim_std > 0 else 1.0)
    vol_norm = (vol - self.vol_mean) / (self.vol_std if self.vol_std > 0 else 1.0)

    return np.column_stack((eff_dim_norm, vol_norm))

  def transform_features(self, sector_returns, market_ohlc, eff_dim_lookback=60):
    if self.s_mean is None:
        raise ValueError("Classifier must be trained before calling transform.")

    s_r_log = np.log(1.0 + sector_returns).values
    s_r_norm = (s_r_log - self.s_mean) / self.s_std
    eff_dim_series = self._get_rolling_eff_dim(s_r_norm, lookback=eff_dim_lookback)
    vol = self.garman_klass_vol(market_ohlc)

    eff_dim_norm = (eff_dim_series - self.eff_dim_mean) / (self.eff_dim_std if self.eff_dim_std > 0 else 1.0)
    vol_norm = (vol - self.vol_mean) / (self.vol_std if self.vol_std > 0 else 1.0)

    return np.column_stack((eff_dim_norm, vol_norm))

  def train_classifier(self, returns, market):
    self._train_returns = returns
    self._train_market = market
    X = self.fit_transform_features(returns, market)
    self.fit(X)

  def generate_blend_params(self, returns, benchmark):
    probs = self.get_regime_probs(returns.iloc[-1], benchmark.iloc[-1])
    state_i = np.argmax(probs)
    transmat = self.hmm.transmat_
    means = self.hmm.means_
    covars = self.hmm.covars_
    posterior_probs = self.predict_proba()
    loglik=self.score_samples()

    cov_type = self.hmm_params.get(cov_type, "diag")
    transmat_safe = np.copy(transmat)
    for idx, row in enumerate(transmat_safe):
      row_sum = np.sum(row)
      if row_sum == 0 or np.isnan(row_sum):
        transmat_safe[idx] = np.full(len(row), 1.0 / len(row))
      else:
        transmat_safe[idx] = row / row_sum

    prob_next_states = transmat_safe[state_i]

    n_components, K_assets = means.shape

    blend_er = prob_next_states @ means

    blend_cov = np.zeros((K_assets, K_assets))
    blend_cov = np.zeros((K_assets))

    for s in range(n_components):
      if cov_type == "full":
        state_cov = covars[s]
      elif cov_type == "diag":
        state_cov = np.diag(covars[s])
      else:
        raise ValueError(f"Unsupported covariance type: {cov_type}")

      shift = (means[s] - blend_er).reshape(-1, 1)

      shift_unc = shift @ shift.T

      blend_cov += prob_next_states[s] * (state_cov + shift_unc)
      blend_mean += prob_next_states[s] * means[s]

    try:
      loglik_flat = np.asarray(loglik).ravel()
      last_ll = loglik_flat[-1]
      ll_means = np.mean(loglik_flat[:-5]) if len(loglik_flat) > 5 else np.mean(loglik_flat)
      loglik_penalty = np.exp(np.maximum(0.0, ll_means - last_ll))

      blend_cov *= loglik_penalty

    except Exception:
      pass

    return blend_cov, blend_mean

  def get_regime_probs(self, returns, market):
    X = self.transform_features(returns, market)
    return super().get_regime_probs(X)

  def score_samples(self):
    if not self.trained:
      raise ValueError("HMM Classifier not trained")
    if self._train_returns is None or self._train_market is None:
      raise ValueError(
          "No training data cached on this classifier -- "
          "call train_classifier(data, benchmark) first."
      )
    X = self.transform_features(self._train_returns, self._train_market)
    return self.hmm.score_samples(X)

  def predict_proba(self):
    if not self.trained:
      raise ValueError("HMM Classifier not trained")
    if self._train_returns is None or self._train_market is None:
      raise ValueError(
          "No training data cached on this classifier -- "
          "call train_classifier(data, benchmark) first."
      )
    X = self.transform_features(self._train_returns, self._train_market)
    return self.hmm.predict_proba(X)

  def score_bic(self, X):
    T, n_features  = X.shape
    n_components   = self.hmm.n_components
    log_likelihood = self.hmm.score(X) * T
    k = n_components * (n_components - 1) + 2 * (n_components * n_features)
    return k * np.log(T) - 2 * log_likelihood

  def expected_regime_duration(self):
    diag = np.clip(np.diag(self.hmm.transmat_), 1e-10, 1.0 - 1e-4)
    return 1.0 / (1.0 - diag)

In [33]:
class Optimizer:
  def __init__(
      self,
      optim_params,
      debug:bool=False,
      **kwargs
  ):
    self.debug = debug
    self.es_prct = optim_params.es_prct
    self.turnover_penalty = optim_params.turnover_penalty
    self.risk_pen = optim_params.risk_penalty
    self.tail_penalty = optim_params.tail_penalty

  def optimize_lambda(self, p, A, b, k_eq, k_ineq):
    constraints = []
    K = k_eq + k_ineq
    lambda_var = cp.Variable(K)

    if k_ineq > 0:
        constraints.append(lambda_var[k_eq:] >= 0)

    exp_terms = -lambda_var @ A + np.log(p)
    obj_fn = cp.log_sum_exp(exp_terms) + lambda_var @ b

    prob = cp.Problem(cp.Minimize(obj_fn), constraints)
    prob.solve(solver=cp.ECOS)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimization failed. Status: {prob.status}")

    return lambda_var.value

  def optimize_w(self, returns, mean, covariance, w_prev=None):
    R = returns.values
    S, N = R.shape
    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    mu = mean.values
    cov = covariance.values

    es = zeta + (1 / ((1-self.es_prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0.0,
        w <= 0.40
    ]

    ex_r = mu @ w

    if w_prev is not None:
      turnover_penalty = self.turnover_penalty * cp.sum((w - w_prev)**2)

    else:
      turnover_penalty = 0

    risk_term = cp.quad_form(w, cov)
    obj_fn = cp.Maximize(ex_r - risk_term - es - turnover_penalty)

    prob = cp.Problem(obj_fn, constraints)
    prob.solve(solver=cp.CLARABEL)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"GMV optimization failed: {prob.status}")

    return w.value

In [36]:
class EntropyPooling(Optimizer):
  def __init__(self, optim_params, base:str="USD", debug:bool=False, **kwargs):
    self.debug = debug
    self.base = base
    super().__init__(debug=debug, base=base, optim_params=optim_params, **kwargs)

  def _add(self, A, b, vtype, direction="le"):
    if vtype == "ineq":
      if direction == "ge":
        self.A_ineq.append(-A)
        self.b_ineq.append(-b)
      else:
        self.A_ineq.append(A)
        self.b_ineq.append(b)
    elif vtype == "eq":
      self.A_eq.append(A)
      self.b_eq.append(b)

  def _compute_window_returns(self, returns, window):
    if window <= 1:
      return returns

    log_r = np.log1p(returns)
    roll_sum = log_r.rolling(window).sum()
    window_returns = np.expm1(roll_sum).dropna()

    return window_returns

  def get_views(self, views, returns, p):
    self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = [], [], [], []

    R = returns.values
    for view in views:
      if view["view"] == "mean":
        idx = returns.columns.get_loc(view["ticker"])

        self._add(
          R[:, idx],
          view["value"],
          view["type"],
          view.get("direction", "le")
        )

      elif view["view"] == "volatility":
        idx = returns.columns.get_loc(view["ticker"])
        R_ = R[:, idx]
        mu = np.sum(p*R_)
        var = (R_ - mu)**2

        self._add(
          var,
          view["value"]**2,
          view["type"],
          view.get("direction", "le")
        )

      elif view["view"] == "relative":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        A_rel = R[:, idx1] - R[:, idx2]
        self._add(
          A_rel,
          view["value"],
          view["type"],
          view.get("direction", "le")
        )

      elif view["view"] == "correlation":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        mu_1 = np.sum(p*R[:, idx1])
        mu_2 = np.sum(p*R[:, idx2])

        vol_1 = np.sqrt(np.sum(p*(R[:, idx1] - mu_1)**2))
        vol_2 = np.sqrt(np.sum(p*(R[:, idx2] - mu_2)**2))
        rho = view["value"]
        cov_target = rho*vol_1*vol_2

        A_corr = (R[:, idx1] - mu_1)*(R[:, idx2] - mu_2)
        self._add(
          A_corr,
          cov_target,
          view["type"],
          view.get("direction", "le")
        )

  def solve_entropy_pooling(
    self,
    R,
    blend_cov_mtx=None,
    views=None,
    p=None,
    window=1
  ):
    R_calc = self._compute_window_returns(R, window)
    S, N = R_calc.shape

    if p is None:
      p = np.ones(S) / S
    else:
      p = np.asarray(p)
      p = p[-S:]
      p = p / p.sum()

    if views is not None:
        self.get_views(views, R_calc, p)
    else:
      self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = None, None, None, None

    A_list, b_list = [], []
    k_eq = 0

    if self.A_eq is not None and self.b_eq is not None and len(self.A_eq) > 0:
      A_eq_clean = np.atleast_2d(self.A_eq)
      b_eq_clean = np.atleast_1d(self.b_eq)
      A_list.append(A_eq_clean)
      b_list.append(b_eq_clean)
      k_eq = A_eq_clean.shape[0]

    k_ineq = 0
    if (
          self.A_ineq is not None
          and self.b_ineq is not None
          and len(self.A_ineq) > 0
      ):
      A_ineq_clean = np.atleast_2d(self.A_ineq)
      b_ineq_clean = np.atleast_1d(self.b_ineq)
      A_list.append(A_ineq_clean)
      b_list.append(b_ineq_clean)
      k_ineq = A_ineq_clean.shape[0]


    if A_list:
      A = np.vstack(A_list)
      b = np.concatenate(b_list)
      opt_lambda = self.optimize_lambda(p, A, b, k_eq, k_ineq)
      q = p * np.exp(-opt_lambda @ A)
      q /= np.sum(q)
    else:
      q = p

    mu_post = q @ R_calc.values
    R_dev = R_calc.values - mu_post
    cov_post = (R_dev.T * q) @ R_dev

    mu_post = pd.Series(mu_post, index=R_calc.columns)
    cov_post = pd.DataFrame(
        cov_post,
        index=R_calc.columns,
        columns=R_calc.columns
    )

    return mu_post, cov_post


  def optimize_portfolio(
      self,
      returns,
      blend_cov_mtx,
      views,
      p=None,
      window=1,
      w_prev=None,
      round_weights=False
    ):
    S, N = returns.shape

    p = p if p is not None else np.ones(S)/S
    print("---------------------- Entropy Pooling ----------------------")
    mu_post, cov_post = self.solve_entropy_pooling(
      returns, blend_cov_mtx, views, p, window=window
    )
    print(f"posterior mean: \n {mu_post}")
    print(f"posterior covariance: \n {cov_post}")

    print("------------------ Optimizing Portfolio w -------------------")
    w_raw = self.optimize_w(returns, mu_post, cov_post, w_prev=w_prev)

    w_opt = self.apply_lot_sizing(w_raw, returns.iloc[-1]) if round_weights else w_raw

    return w_opt, returns.columns

In [37]:
class Portfolio(DataStore, Filter, RegimeClassifier, EntropyPooling):
  def __init__(
      self,
      hmm_params:HMMParams,
      optimizer_params:OptimParams,
      filter_params:FilterParams=None,
      mc_params:MCParams=None,
      blend_method:str="shrinkage", # shrinkage/monte_carlo
      debug:bool=False,
      **kwargs
    ):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug
    self.filter_params = filter_params
    self.mc_params = mc_params

  def get_data(self, universe, start, end, benchmark_ticker="^GSPC"):
    tickers = list(universe.values())
    data, benchmark = self._get_data(
        tickers=tickers,
        benchmark=benchmark_ticker
    )

    return data, benchmark

  def filter_universe(
    self,
    universe:dict,
    data:pd.DataFrame,
    benchmark:pd.DataFrame,
    cpi:pd.DataFrame,
    filter_params:dict
  ):
    returns = {}
    for ticker in data.columns:
      returns[ticker] = data[ticker]["Close"].pct_change().dropna()

    returns = pd.DataFrame(returns)


  def get_regime(self, returns, benchmark, prev_classifier=None):
    data = returns.copy()
    if self.filter_params:
      cpi = self.get_cpi()
      qualified = self.filter_universe(returns, benchmark, cpi)
      data = data.loc[:, qualified]

    classifier = self.regime_classifier_cls(hmm_params=self.hmm_params, debug=self.debug)
    if len(self.results) > 0 and prev_classifier:
      classifier.hmm.transmat_ = prev_classifier.transmat_.copy()
      classifier.hmm.means_ = prev_classifier.means_.copy()
      classifier.hmm.covars_ = prev_classifier._covars_.copy()
      classifier.hmm.startprob_ = prev_classifier.startprob_.copy()
      classifier.hmm.init_params = ""

    classifier.train_classifier(data, benchmark)
    regime_probs = classifier.get_regime_probs(data, benchmark)

    classifier.train_classifier(returns, benchmark)
    regime_probs = classifier.get_regime_probs(returns, benchmark)

    return classifier, regime_probs

  def simulate_paths

  def generate_portfolio(self, data, benchmark, w_prev=None, prev_classifier=None):
    returns = {}
    for ticker in data.columns:
      returns[ticker] = data[ticker]["Close"].pct_change().dropna()

    returns = pd.DataFrame(returns)

    classifier, regime_probs = self.get_regime(returns, benchmark, prev_classifier)

    X_train = classifier.fit_transform_features(returns, benchmark)
    bic = classifier.score_bic(X_train)
    expected_duration = classifier.expected_regime_duration()

    high_vol_state = int(np.argmax(classifier.hmm.means_[:, 1]))

    blend_cov, blend_mean = self.generate_blend_params(
        returns=returns,
        benchmark=benchmark,
    )

    sim_paths = self.simulate_paths(
        S = self.S,
        N = blend_mean.shape[0],
        mean = blend_mean,
        cov = blend_cov
    )














In [ ]:
ptf = Portfolio(

)